<h1 align="left">
  <picture>
    <source media="(prefers-color-scheme: dark)" srcset="https://autonomousvision.github.io/py123d/_static/123D_logo_transparent_white.svg" width="500">
    <source media="(prefers-color-scheme: light)" srcset="https://autonomousvision.github.io/py123d/_static/123D_logo_transparent_black.svg" width="500">
    <img alt="Logo" src="https://autonomousvision.github.io/py123d/_static/123D_logo_transparent_black.svg" width="500">
  </picture>
  <h2 align="left">123D: Scene Tutorial</h1>
</h1>

In [ ]:
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np

from py123d.api import SceneAPI, SceneFilter

## 1.1 Download Demo Logs

You can download demo logs for 123D as described in the [documentation](https://autonomousvision.github.io/py123d/installation/). After the installation and download, you can start with the tutorial.

## 1.2 Create Scenes by filtering the datasets



The logs store continuous driving recordings. Scenes in 123D are sequences that are extracted from a log, e.g. given a predefined duration and history.

In the example below, we filter some scenes from all logs with 8 second duration and 8 seconds temporal distance (making the scenes non-overlapping). If `None` is passed to the duration, the scene will contain the complete log.

This `SceneFilter` object is passed to a `SceneBuilder` object to query `SceneAPI`'s from the dataset.





In [ ]:
from py123d.api import get_filtered_scenes
from py123d.common.execution import ThreadPoolExecutor
from py123d.datatypes import CameraID

scene_filter = SceneFilter(
    datasets=["nuscenes-mini", "av2-sensor"],
    split_names=None,
    log_names=None,
    scene_uuids=None,
    target_iteration_duration_s=0.5,
    future_duration_s=10.0,
    history_duration_s=2.0,
    timestamp_threshold_s=0.1,
    # required_scene_modalities=["ego_state_se3"],
    # shuffle=True,
)
scenes = get_filtered_scenes(scene_filter, executor=ThreadPoolExecutor())

dataset_splits = set(scene.log_metadata.split for scene in scenes)
print(f"Found {len(scenes)} scenes from {len(dataset_splits)} datasplits:")
for split in dataset_splits:
    print(f" - {split}")

In [ ]:
scenes_by_split = {}
for scene in scenes:
    split_name = scene.log_metadata.split
    scenes_by_split.setdefault(split_name, []).append(scene)

print(f"Grouped scenes into {len(scenes_by_split)} splits.")
for split_name, split_scenes in scenes_by_split.items():
    print(f"{split_name}: {len(split_scenes)} scenes")

In [ ]:
scenes_by_split = {}
for scene in scenes:
    split_name = scene.log_metadata.split
    scenes_by_split.setdefault(split_name, []).append(scene)

print(f"Grouped scenes into {len(scenes_by_split)} splits.")
for split_name, split_scenes in scenes_by_split.items():
    print(f"{split_name}: {len(split_scenes)} scenes")

In [ ]:
# wod-motion_val: 2288 scenes
# wod-motion_train: 3968 scenes
# nuplan-mini_train: 17909 scenes
# nuplan-mini_val: 3754 scenes
# nuplan-mini_test: 4224 scenes
# nuscenes-interpolated-mini_train: 153 scenes
# av2-sensor_train: 120 scenes
# nuscenes-interpolated-mini_val: 38 scenes
# wod-perception_val: 38 scenes

key = list(scenes_by_split.keys())[0]
print(key)


np.random.choice(scenes_by_split["av2-sensor_train"]).get_scene_metadata().iteration_duration_s

## 1.2 Inspecting the Scene

Let's inspect a random scenefrom our dataset.

A scene has several different metadata objects attached to it:

`SceneMetadata`: Information how the scene was extracted from the log. Each timestep in the log has universally unique identifier (UUID). The UUID of the initial timestep also serves as identifier for scene filtering.

In [ ]:
scene: SceneAPI = np.random.choice(scenes)  # type: ignore
scene_metadata = scene.scene_metadata
print(scene_metadata)
print("\nInitial UUID:", scene_metadata.initial_uuid)
print("Number of iterations:", scene_metadata.num_future_iterations)
print("Number of history iterations:", scene_metadata.num_history_iterations)
print("Duration (s):", scene_metadata.future_duration_s)
print("Iteration duration (s):", scene_metadata.iteration_duration_s)

`LogMetadata`: Information of the log the scene was extracted from. This object also includes data about the map (if available), or static information of the ego vehicle, e.g. the included sensors and vehicle parameters

In [ ]:
log_metadata = scene.get_log_metadata()
log_metadata

`MapMetadata`: If the map is available, this object includes information about the location, wether the map is 3D (`map_has_z`), of the the map is defined per log (`map_is_per_log`).

In [ ]:
map_metadata = scene.get_map_metadata()
map_metadata

## 1.3 Retrieving data from the `SceneAPI`

Different datasets might provide different modalities. In general, you can load data using a `scene.get_modality_at_iteration(iteration=...)`

If a modality is not available, the return will be `None`. The `Timestamp` is the only modality that is strictly required to be available in a `Scene

Let's look at some examples:


### 1.3.1 `Timestamp`

Current time step in microseconds.

In [ ]:
from py123d.datatypes.time import Timestamp

iteration = 0
timestamp: Timestamp = scene.get_timestamp_at_iteration(iteration=iteration)
print(f"Time at iteration {iteration}:", timestamp)

In [ ]:
import time

start = time.time()

bboxes = scene.get_box_detections_se3_at_iteration(iteration=iteration)

# 0.0004315376

end = time.time()
print(f"Time to get box detections at iteration {iteration}: {end - start:.10f} seconds")

### 1.3.2 `EgoStateSE3` 
State of the ego vehicle in 3D with location and orientation.

In [ ]:
ego_state_se3 = scene.get_ego_state_se3_at_iteration(iteration=iteration)

if ego_state_se3 is not None:
    print("Ego parameters\t", ego_state_se3.metadata)

    # The ego vehicles coordinate system is defined by it's rear-axle / IMU location.
    print("Rear axle location:\t", ego_state_se3.rear_axle_se3.point_3d)
    print("Rear axle orientation:\t", ego_state_se3.rear_axle_se3.quaternion)

    # You can also use the center pose
    print("Center location:\t", ego_state_se3.center_se3.point_3d)
    print("Center orientation:\t", ego_state_se3.center_se3.quaternion)

### 1.3.3 `BoxDetectionsSE3`

Object that contains all bounding boxes in the current time step

In [ ]:
from py123d.datatypes.detections.box_detections import BoxDetectionsSE3

box_detections_se3: Optional[BoxDetectionsSE3] = scene.get_box_detections_se3_at_iteration(iteration=iteration)

if box_detections_se3 is not None:
    print(f"Number of boxes:{len(box_detections_se3)}")

    if len(box_detections_se3) > 0:
        box_detection = box_detections_se3[0]
        print("\nFirst box:")
        print("Dataset Label:\t", box_detection.attributes.label)
        print("Default Label:\t", box_detection.attributes.default_label)
        print("Parameters:\t", box_detection.bounding_box_se3)

### 1.3.4 `Camera`
Object containing the camera observation with a pinhole model.

In [ ]:
from typing import Optional

from py123d.datatypes import Camera

available_camera_ids = scene.available_camera_ids
print("Available pinhole camera types:\t", available_camera_ids)

random_camera_id = np.random.choice(available_camera_ids + [CameraID.PCAM_F0])  # type: ignore
random_camera: Optional[Camera] = scene.get_camera_at_iteration(iteration=iteration, camera_id=random_camera_id)


# NOTE: If a camera is not available, the return will be None
if random_camera is not None:
    print(f"\nCamera ID:\t{random_camera_id}")

    print(f"Image shape:\t{random_camera.image.shape}")
    print(f"Model:\t{random_camera.metadata.camera_model}")

    plt.imshow(random_camera.image)
    plt.title(f"Camera ID: {random_camera_id}")
    plt.show()

### 1.3.5 `Lidar`
Object containing a point cloud of a single laser scanner.

In [ ]:
from py123d.datatypes.sensors import LidarID

available_lidar_ids = scene.available_lidar_ids
print("Available Lidar types:\t", available_lidar_ids)


random_lidar_id = np.random.choice(available_lidar_ids + [LidarID.LIDAR_TOP])  # type: ignore
random_lidar = scene.get_lidar_at_iteration(iteration=iteration, lidar_id=random_lidar_id)

if random_lidar is not None:
    point_cloud_3d = random_lidar.point_cloud_3d
    point_cloud_features = random_lidar.point_cloud_features

    print(f"\nLidar ID:\t{random_lidar_id}")
    print(f"Point cloud:\t shape {random_lidar.point_cloud_3d.shape}, dtype: {random_lidar.point_cloud_3d.dtype}")
    print("Features:")
    if random_lidar.point_cloud_features is not None:
        for key in random_lidar.point_cloud_features.keys():
            print(
                f"\t{key}:\t shape: {random_lidar.point_cloud_features[key].shape}\t dtype: {random_lidar.point_cloud_features[key].dtype}"
            )
    else:
        print("\tNo point cloud features available.")

    xy = random_lidar.xy

    plt.scatter(xy[:, 0], xy[:, 1], s=0.1, alpha=0.25, c="black")
    plt.title(f"Lidar ID: {random_lidar_id}")
    plt.xlabel("X-forward [m]")
    plt.ylabel("Y-left [m]")
    plt.axis("equal")

    range_limit = 100  # meters
    plt.xlim(-range_limit, range_limit)
    plt.ylim(-range_limit, range_limit)
    plt.show()

### 1.3.6 `MapAPI`

The `MapAPI` can get retrieved from a scene directly. If the map is available, we plot the map with our default plotting function.
For further information, you can visit the map or visualization tutorial.

In [ ]:
from py123d.api import MapAPI
from py123d.geometry import Point2D
from py123d.visualization.matplotlib.observation import add_default_map_on_ax
from py123d.visualization.matplotlib.utils import add_non_repeating_legend_to_ax


def simple_map_visualization(map_api: MapAPI, center_2d: Point2D, map_radius: float = 100.0):
    """Helper to plot the map using matplotlib.

    :param map_api: The MapAPI to visualize
    :param center_2d: The center point of the map visualization
    :param map_radius: The radius around the center point to visualize
    """

    fsize = 8
    _, ax = plt.subplots(figsize=(fsize, fsize))
    add_default_map_on_ax(ax, map_api=map_api, point_2d=center_2d, radius=map_radius)
    add_non_repeating_legend_to_ax(ax)
    ax.set_aspect("equal")
    ax.set_xlim(center_2d.x - map_radius, center_2d.x + map_radius)
    ax.set_ylim(center_2d.y - map_radius, center_2d.y + map_radius)
    plt.show()


map_api = scene.get_map_api()
if map_api is not None:
    ego_state_se3 = scene.get_ego_state_se3_at_iteration(iteration=iteration)
    if ego_state_se3 is not None:
        center_2d = ego_state_se3.center_se3.point_2d
    else:
        center_2d = Point2D.from_array(np.array([0.0, 0.0]))

    print("\nMapAPI is available.")
    print("Map Metadata:", map_api.map_metadata)
    simple_map_visualization(map_api=map_api, center_2d=center_2d)

### 1.3.7 Others:

You can find further modalities in the documentation of [`SceneAPI`](todo)